# Data Science Fundamentals Assessment
**Submitted by:** Rajat | **Program:** Yuva Intern | **Date:** 27 Nov 2026

Covers: Python basics, data types, data cleaning, statistical foundations, and industry best practices.

## 1. Setup and Reproducibility
Set a random seed and import the standard data science libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)   # reproducibility
pd.set_option("display.precision", 3)

## 2. Python Basics

### 2.1 Variables and built-in types

In [ ]:
age = 21            # int
gpa = 8.4           # float
name = "Rajat"      # str
is_student = True   # bool
for v in (age, gpa, name, is_student):
    print(repr(v), "->", type(v).__name__)

### 2.2 Collections: list, tuple, set, dict

In [ ]:
scores = [45, 50, 55, 58, 65, 70, 74, 80, 88, 95]   # list (ordered, mutable)
point = (28.6, 77.2)                                  # tuple (immutable)
unique_genders = set(["M", "F", "F", "M"])            # set (unique values)
student = {"id": 1, "name": "Asha", "score": 45}      # dict (key-value)

print(len(scores), point, unique_genders, student["name"])

### 2.3 Control flow, functions and comprehensions

In [ ]:
def grade(s):
    """Return a letter grade for a numeric score."""
    if s >= 80:
        return "A"
    elif s >= 60:
        return "B"
    return "C"

grades = [grade(s) for s in scores]       # list comprehension
print(grades)

total = 0
for s in scores:
    total += s
print("Total:", total, "| Average:", total / len(scores))

### 2.4 NumPy arrays (vectorised operations)

In [ ]:
arr = np.array(scores)
print("Mean:", arr.mean(), "| Max:", arr.max())
print("Scaled to 0-1:", np.round((arr - arr.min()) / (arr.max() - arr.min()), 2))

## 3. Dataset and Data Types
Sample dataset of 10 students. One score is deliberately missing to practise cleaning.

In [ ]:
df = pd.DataFrame({
    "student_id":     range(1, 11),
    "hours_studied":  [2, 3, 4, 5, 6, 7, 8, 9, 10, 12],
    "attendance_pct": [60, 65, 70, 72, 80, 82, 88, 90, 95, 98],
    "gender":         ["M", "F", "F", "M", "F", "M", "F", "M", "F", "M"],
    "score":          [45, 50, 55, 58, 65, 70, 74, 80, 88, 95],
})
df.loc[3, "score"] = np.nan    # introduce a missing value
df

### 3.1 Statistical data types
| Type | Sub-type | Example in this dataset |
|---|---|---|
| Quantitative | Discrete | `hours_studied` (whole hours) |
| Quantitative | Continuous | `score`, `attendance_pct` |
| Qualitative | Nominal | `gender` |
| Qualitative | Ordinal | letter `grade` (A > B > C) |

In [ ]:
print(df.dtypes)
print()
df.info()

In [ ]:
# Convert gender to categorical and create an ordered categorical grade
df["gender"] = df["gender"].astype("category")
print(df["gender"].dtype)

## 4. Data Cleaning

In [ ]:
print("Shape:", df.shape)
print("\nMissing values per column:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

The `score` column has 1 missing value. Median imputation is used because it is robust to outliers.

In [ ]:
df["score"] = df["score"].fillna(df["score"].median())
print("Missing after cleaning:", df["score"].isna().sum())
df["grade"] = pd.Categorical(df["score"].apply(grade), categories=["C", "B", "A"], ordered=True)
df

## 5. Statistical Foundations

### 5.1 Descriptive statistics

In [ ]:
s = df["score"]
q1, q3 = s.quantile([0.25, 0.75])
iqr = q3 - q1
summary = pd.Series({
    "Mean": s.mean(), "Median": s.median(), "Mode": s.mode()[0],
    "Std dev": s.std(), "Variance": s.var(), "Q1": q1, "Q3": q3,
    "IQR": iqr, "Skewness": stats.skew(s),
})
summary.round(3)

In [ ]:
df.describe()

### 5.2 Outlier detection (1.5 x IQR rule)

In [ ]:
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = df[(df["score"] < lower) | (df["score"] > upper)]
print(f"Valid range: [{lower:.1f}, {upper:.1f}]")
print("Outliers found:", len(outliers))

### 5.3 Distribution visualisation

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(df["score"], bins=5, edgecolor="black")
ax[0].set_title("Histogram of scores"); ax[0].set_xlabel("Score"); ax[0].set_ylabel("Count")
ax[1].boxplot(df["score"], vert=False)
ax[1].set_title("Boxplot of scores"); ax[1].set_xlabel("Score")
plt.tight_layout(); plt.show()

### 5.4 Correlation

In [ ]:
corr = df[["hours_studied", "attendance_pct", "score"]].corr()
print(corr.round(3))

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(3)); ax.set_xticklabels(corr.columns, rotation=30)
ax.set_yticks(range(3)); ax.set_yticklabels(corr.columns)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
plt.colorbar(im); plt.title("Correlation heatmap"); plt.show()

### 5.5 Hypothesis testing

In [ ]:
# Pearson correlation test: H0 = no linear relationship between hours studied and score
r, p = stats.pearsonr(df["hours_studied"], df["score"])
print(f"r = {r:.3f}, p-value = {p:.2e}")
print("Significant at 0.05" if p < 0.05 else "Not significant")

In [ ]:
# Two-sample t-test: do male and female scores differ?
m = df.loc[df["gender"] == "M", "score"]
f = df.loc[df["gender"] == "F", "score"]
print("Mean M:", m.mean(), "| Mean F:", f.mean())
t, p = stats.ttest_ind(m, f, equal_var=False)
print(f"t = {t:.3f}, p-value = {p:.3f}")
print("Significant at 0.05" if p < 0.05 else "Not significant: difference may be due to chance")

### 5.6 Linear regression (simple)

In [ ]:
res = stats.linregress(df["hours_studied"], df["score"])
print(f"score = {res.slope:.2f} * hours + {res.intercept:.2f}   (R^2 = {res.rvalue**2:.3f})")

plt.scatter(df["hours_studied"], df["score"], label="Data")
x = np.array([df["hours_studied"].min(), df["hours_studied"].max()])
plt.plot(x, res.slope * x + res.intercept, color="red", label="Fit")
plt.xlabel("Hours studied"); plt.ylabel("Score"); plt.legend(); plt.title("Hours vs Score"); plt.show()

## 6. Industry Best Practices
- **Reproducibility:** fixed random seed, pinned versions in `requirements.txt`, virtual environments.
- **Version control:** Git/GitHub with clear commits and a README.
- **Clean code:** descriptive names, PEP 8, docstrings, small reusable functions.
- **Avoid data leakage:** split train/test first, fit imputers and scalers on training data only.
- **Explore before modelling:** EDA and visualisation first.
- **Right metrics:** F1/recall for imbalanced classes instead of plain accuracy.
- **Ethics and privacy:** protect personal data, check for bias, report limitations.
- **Communication:** clear charts and plain-language conclusions.

## 7. Conclusion
- Cleaned data (1 missing score filled with median); no outliers under the IQR rule.
- Scores are roughly symmetric (mean 69.2, median 70.0, skewness about 0.03).
- Hours studied vs score: r = 0.979, p < 0.05, a very strong and significant positive relationship.
- Gender difference in scores is not statistically significant in this small sample.
- **Limitations:** only 10 observations, and correlation does not imply causation. Hours and attendance are also highly correlated (multicollinearity).